In [7]:
import pandas as pd
import json
import os
import re

input_file = "tokenize_job/datasets/binary_single_train/CJPE_ext_SCI_HCs_Tribunals_daily_orders_single.csv"
df = pd.read_csv(input_file, sep=None, engine="python", on_bad_lines="skip")

# ✅ Print row/column count immediately
print("CSV loaded successfully!")
print(f"Total rows: {len(df)}, Total columns: {len(df.columns)}")
print("\nFirst 5 rows:")
print(df.head())

output_file = "dataset_single_lora_final.jsonl"

# Define court type mapping based on filename patterns
def get_court_type(filename):
    filename = filename.lower()
    if "supreme_court" in filename:
        return "Supreme Court"
    elif "high_court" in filename or "hc_" in filename:
        return "High Court"
    elif "district_court" in filename:
        return "District Court"
    elif "tribunal" in filename:
        return "Tribunals"
    elif "daily_order" in filename:
        return "Daily Order"
    else:
        return "Unknown"

# Function to clean text into a single line
def clean_text(text):
    # Convert to string, remove line breaks and multiple spaces
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)  # Replace all whitespace (newlines, tabs, multiple spaces) with a single space
    return text

with open(output_file, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        label = str(row["label"]).strip().lower()
        output = "Accepted" if label in ["accept", "accepted", "1"] else "Rejected"
        filename = str(row["filename"])
        court_type = get_court_type(filename)
        record = {
            "instruction": "Given the text of an Indian court judgment, predict the final decision as either Accepted or Rejected.",
            "input": clean_text(row["text"]),
            "output": output,
            "metadata": {
                "filename": filename,
                "court_type": court_type
            }
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"\n✅ Conversion complete! Saved as {output_file}")

print("\n📊 Final dataset stats:")
print(f"Rows written: {len(df)}")

print("\nPreview of JSONL file:")
with open(output_file, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f.readline().strip())

if os.path.exists(output_file):
    print(f"\n📂 File ready at: {os.path.abspath(output_file)}")


CSV loaded successfully!
Total rows: 300803, Total columns: 3

First 5 rows:
                                            filename  \
0  Calcutta_High_Court_Appellete_Side_2008_2020_2...   
1                           Bombay_HC_BomHC_1937_179   
2                              karnataka_HC_1998_425   
3                            Bombay_HC_BomHC_1994_60   
4                                Madras_HC_2017_4649   

                                                text  label  
0  Judgment on 18th April, 2008.Bhaskar J.These t...      0  
1  Beaumont, C.J.This is an appeal from a decisio...      0  
2  The appellant has filed this appeal assailing ...      0  
3  The petitioner, Forbes Forbes Campbel and.Comp...      1  
4  The writ petitioner was allotted plot No.60 fo...      1  

✅ Conversion complete! Saved as dataset_single_lora_final.jsonl

📊 Final dataset stats:
Rows written: 300803

Preview of JSONL file:
{"instruction": "Given the text of an Indian court judgment, predict the final d

In [6]:
import pandas as pd
import hashlib
import json

# Paths to your CSV files
file1 = "tokenize_job/datasets/binary_multi_train/CJPE_ext_SCI_HCs_Tribunals_daily_orders_multi.csv"
file2 = "tokenize_job/datasets/binary_single_train/CJPE_ext_SCI_HCs_Tribunals_daily_orders_single.csv"

# ---------- STEP 1: File size + md5 ----------
def file_md5(path):
    hash_md5 = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

print("File1 MD5:", file_md5(file1))
print("File2 MD5:", file_md5(file2))

# ---------- STEP 2: Load into pandas ----------
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

print("\n--- Shape Check ---")
print("File1:", df1.shape, "File2:", df2.shape)
if df1.shape == df2.shape:
    print("✅ Both datasets have the same shape")
else:
    print("❌ Shapes differ")

print("\n--- Column Names Check ---")
if list(df1.columns) == list(df2.columns):
    print("✅ Column names match")
else:
    print("❌ Column names differ")
    print("File1:", df1.columns.tolist())
    print("File2:", df2.columns.tolist())

# ---------- STEP 3: Row content deep check ----------
def hash_row(row):
    return hashlib.md5(json.dumps(row, sort_keys=True).encode()).hexdigest()

hashes1 = df1.apply(lambda r: hash_row(r.to_dict()), axis=1)
hashes2 = df2.apply(lambda r: hash_row(r.to_dict()), axis=1)

print("\n--- Row Content Check ---")
if hashes1.equals(hashes2):
    print("✅ All rows are identical in both datasets")
else:
    print("❌ Datasets differ in content")
    only_in_1 = set(hashes1) - set(hashes2)
    only_in_2 = set(hashes2) - set(hashes1)
    print("Rows only in File1:", len(only_in_1))
    print("Rows only in File2:", len(only_in_2))
    # Save mismatches if needed
    mismatched1 = df1[hashes1.isin(only_in_1)]
    mismatched2 = df2[hashes2.isin(only_in_2)]
    mismatched1.to_csv("diff_file1.csv", index=False)
    mismatched2.to_csv("diff_file2.csv", index=False)
    print("Mismatched rows saved to diff_file1.csv and diff_file2.csv")



File1 MD5: 019f09cabf08ccc3a053213e26e09f49
File2 MD5: fd6d3b5ed933ba760d7c9ca60bb3ac47

--- Shape Check ---
File1: (527208, 3) File2: (301059, 3)
❌ Shapes differ

--- Column Names Check ---
✅ Column names match

--- Row Content Check ---
❌ Datasets differ in content
Rows only in File1: 301562
Rows only in File2: 75413
Mismatched rows saved to diff_file1.csv and diff_file2.csv
